# Datamine WEDGEVOL Process Example

This notebook demonstrates how to configure and run the **`wedgevol`** process wrapper in `dmstudio`.

## Process Description

## WEDGEVOL

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

Evaluate a wedge volume limited by two or three DTMs or wireframe surfaces.

The process requires both wireframe files (**WIRETR1** ,**WIREPT1** and **WIRETR2** ,**WIREPT2** and optionally a third surface **WIRETR3** ,**WIREPT3** ;) and a prototype block model (**PROTO**) as input.

The wedge position parameters (**SRFTYPE1** , **SRFTYPE2** ,optionally **SRFTYPE3**) define the position of the wedge volume relative to each wireframe surface as follows:

* **1** (the wedge is located below the surface,):

* **2** (the wedge is located above the surface.):

The density of the wedge volume material is set using the **DENSITY** parameter. Subcell splitting of the wedge block model cells is controlled by the **SPLITS** parameter, where SPLIT=0 does not split parent block model cells into subcells, **SPLIT** =1 splits the parent cells once, etc. The minimum and maximum elevations of the wedge volume are defined by the **ZMIN** and **ZMAX** parameters respectively.

The output from this process is an (evaluation) results file and optionally the wedge volume block model.

### Input Files:

* **wiretr1** (Wireframe triangle):
  Triangle file of update wireframe surface 1 (DTM).
  Required=Yes

* **wirept1** (Wireframe points):
  Point file of update wireframe surface 1 (DTM).
  Required=Yes

* **wiretr2** (Wireframe triangle):
  Triangle file of update wireframe surface 2 (DTM).
  Required=Yes

* **wirept2** (Wireframe points):
  Point file of update wireframe surface 2 (DTM).
  Required=Yes

* **wiretr3** (Wireframe triangle):
  Triangle file of wireframe surface 3 (DTM).
  Required=No

* **wirept3** (Wireframe points):
  Point file of update wireframe surface 3 (DTM).
  Required=No

* **proto** (Block model):
  Block model prototype.
  Required=Yes

### Output Files:

* **modelou** (Block model):
  Wedge block model.
  Required=No

* **results** (Results file):
  Output evaluation results data file.
  Required=Yes

### Fields:

### Parameters:

* **sftype1**:
  Wedge below or above surface 1. Select one of the following options, with the default
  being 1. 1: Create wedge below. 2: Create wedge above.
  Range=1,2
  Values=1,2
  Default=1
  Required=Yes

* **sftype2**:
  Wedge below or above surface 2. Select one of the following options, with the default
  being 2. 1: Create wedge below. 2: Create wedge above.
  Range=1,2
  Values=1,2
  Default=
  Required=Yes

* **sftype3**:
  Wedge below or above surface 3. Select one of the following options, with the default
  being 2. 1: Create wedge below. 2: Create wedge above.
  Range=1,2
  Values=1,2
  Default=2
  Required=No

* **density**:
  Density of filled volumes.
  Range=0.001,99999
  Values=
  Default=1
  Required=Yes

* **splits**:
  Subcell splitting of internal wedge block model.
  Range=0,3
  Values=
  Default=0
  Required=Yes

* **zmin**:
  Minimum elevation of wedge volume.
  Range=
  Values=
  Default=-9999999
  Required=Yes

* **zmax**:
  Maximum elevation of wedge volume.
  Range=
  Values=
  Default=9999999
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('wedgevol')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute wedgevol
print("Running wedgevol...")
dm_cmd.wedgevol(
    wiretrs_i=['t_SurfaceTriangles'],  # required
    wirepts_i=['t_SurfacePointsPt'],  # required
    proto_i='t_mod1',  # required
    modelou_o='t_wedgevol_out',  # required
    results_o=['t_wedgevol_out'],  # required
    sftypes_f=['optional'],  # required
    density_p='required_val',  # required
    splits_p='required_val',  # required
    zmin_p='required_val',  # required
    zmax_p='required_val',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("wedgevol execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_wedgevol_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")